# Time Travel Rewind — GRPO Training with Unsloth

Trains **Qwen3-14B** (4-bit via Unsloth) on the **Time Travel Rewind** OpenEnv environment using a custom GRPO loop.

**The puzzle:** the agent navigates `start → door → path-1 → path-2 → sign`, reads a passcode from the sign, then *branches* (rewinds time) back to the door and uses the passcode to unlock it — all within a limited action budget.

**Runtime required:** GPU (T4 minimum). Set `Runtime → Change runtime type → T4 GPU`.

## 1. Install dependencies

In [ ]:
!pip install unsloth --quiet
!pip install 'openenv-core[core]==0.2.1' uvicorn --quiet
!pip install wandb --quiet

## 2. Clone the environment repo

In [ ]:
import os

REPO_URL = "https://huggingface.co/spaces/shubhampatilsd/timetravel_openenv"
REPO_DIR = "/content/timetravel_openenv"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print("Working directory:", os.getcwd())

## 3. Start the OpenEnv server

The environment runs as a local FastAPI server that the training loop connects to over WebSocket.

In [ ]:
import subprocess, sys, time, urllib.request

server_proc = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "server.app:app",
     "--host", "0.0.0.0", "--port", "7860"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)
time.sleep(4)

try:
    urllib.request.urlopen("http://localhost:7860/timetravel/health", timeout=5)
    print("Server is up at http://localhost:7860")
except Exception as e:
    print("Server starting (this is OK):", e)

## 4. Configuration

In [ ]:
# ── Environment ────────────────────────────────────────────────────
ENV_URL = "http://localhost:7860/timetravel"

# ── Model ──────────────────────────────────────────────────────────
MODEL_NAME         = "unsloth/Qwen3-14B-unsloth-bnb-4bit"
MAX_SEQ_LENGTH     = 2048
LORA_RANK          = 4

# ── Training ───────────────────────────────────────────────────────
NUM_TRAIN_STEPS    = 300
EPISODES_PER_STEP  = 4   # independent episode groups per gradient step
NUM_GENERATIONS    = 4   # rollouts per group (GRPO group size)
TEMPERATURE        = 0.7
LR                 = 2e-4
WEIGHT_DECAY       = 0.01
MAX_GRAD_NORM      = 1.0

# ── Episode ────────────────────────────────────────────────────────
MAX_EPISODE_STEPS  = 12  # max actions per episode
GEN_MAX_NEW_TOKENS = 64  # max tokens per action

# ── Logging / saving ───────────────────────────────────────────────
EVAL_EVERY         = 25
EVAL_EPISODES      = 10
SAVE_EVERY         = 100
OUTPUT_DIR         = "/content/runs/timetravel"
WANDB_PROJECT      = "timetravel"
WANDB_RUN_NAME     = "qwen3-14b-timetravel-colab"
SEED               = 3407

## 5. Environment helpers

In [ ]:
import json, re
from typing import Optional

SYSTEM_PROMPT = """You are an agent navigating a time-travel puzzle.

Map: start -> door -> path-1 -> path-2 -> sign

Goal: unlock the door at `door` with the passcode shown at `sign`.
Budget: limited steps total.

The branch command rewinds time by N steps. After the rewind your chat history is erased back to that point — you will NOT remember what you saw. The only thing that survives is the instruction field, which becomes your temporal note. You MUST embed the exact passcode in the instruction, e.g. "Use passcode AURORA-314 at door". After a branch, read your temporal note to get the passcode.

Output exactly one JSON object and nothing else. No explanation outside the JSON.

Valid formats:
  {"thinking":"...","action":"move_forward","args":{}}
  {"thinking":"...","action":"move_back","args":{}}
  {"thinking":"...","action":"read_sign","args":{}}
  {"thinking":"...","action":"open_door","args":{"passcode":"..."}}
  {"thinking":"...","action":"branch","args":{"ago":N,"instruction":"Use passcode X at door"}}
  {"thinking":"...","action":"abandon","args":{}}
"""

_JSON_RE = re.compile(r"\{.*?\}", re.DOTALL)

def obs_to_text(obs: dict, step_num: int) -> str:
    lines = [
        f"Step {step_num} | Budget remaining: {obs['budget_remaining']}",
        f"Position: {obs['position']}",
    ]
    if obs.get("temporal_note"):
        lines.append(f"Temporal note from future self: {obs['temporal_note']}")
    if obs.get("message"):
        lines.append(obs["message"])
    return "\n".join(lines)

def parse_action(text: str) -> Optional[dict]:
    for candidate in _JSON_RE.findall(text):
        try:
            p = json.loads(candidate)
            if isinstance(p, dict) and "action" in p:
                return p
        except json.JSONDecodeError:
            pass
    try:
        p = json.loads(text)
        if isinstance(p, dict) and "action" in p:
            return p
    except json.JSONDecodeError:
        pass
    return None

def format_action(action: dict) -> str:
    return json.dumps(action, separators=(",", ":"))

def infer_success(obs: dict) -> bool:
    return bool(obs.get("succeeded", False))

print("Helpers ready.")

## 6. WebSocket environment client

OpenEnv maintains episode state via WebSocket — one connection per episode.

In [ ]:
import sys
sys.path.insert(0, "/content/timetravel_openenv")

from models import TimetravelAction, TimetravelObservation
from openenv.core import EnvClient
from openenv.core.client_types import StepResult


class TimetravelEnv(EnvClient):
    """WebSocket client for the Time Travel Rewind environment."""

    def _step_payload(self, action):
        return {"content": action}  # action is a JSON string

    def _parse_result(self, payload):
        obs_data = payload.get("observation", {})
        observation = TimetravelObservation(
            message=obs_data.get("message", ""),
            position=obs_data.get("position", "start"),
            budget_remaining=obs_data.get("budget_remaining", 0),
            active_timeline_id=obs_data.get("active_timeline_id", 0),
            num_branches=obs_data.get("num_branches", 0),
            read_sign_count=obs_data.get("read_sign_count", 0),
            succeeded=obs_data.get("succeeded", False),
            protocol_violations=obs_data.get("protocol_violations", 0),
            temporal_note=obs_data.get("temporal_note"),
            done=payload.get("done", False),
            reward=payload.get("reward"),
            metadata=obs_data.get("metadata", {}),
        )
        return StepResult(
            observation=observation,
            reward=payload.get("reward"),
            done=payload.get("done", False),
        )

    def _parse_state(self, payload):
        from openenv.core.env_server.types import State
        return State(
            episode_id=payload.get("episode_id"),
            step_count=payload.get("step_count", 0),
        )


def env_reset(env) -> dict:
    result = env.reset()
    obs = result.observation.model_dump()
    obs["done"] = result.done
    obs["reward"] = float(result.reward or 0.0)
    return obs


def env_step(env, action_json: str) -> dict:
    result = env.step(action_json)
    obs = result.observation.model_dump()
    obs["done"] = result.done
    obs["reward"] = float(result.reward or 0.0)
    return obs


# Sanity check
with TimetravelEnv(base_url=ENV_URL) as env:
    obs = env_reset(env)
    print("Reset OK. Position:", obs["position"], "| Budget:", obs["budget_remaining"])

## 7. Load model with Unsloth

In [ ]:
import random, torch
from unsloth import FastLanguageModel

random.seed(SEED)
torch.manual_seed(SEED)

print(f"Loading {MODEL_NAME} ...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    load_in_4bit=True,
    max_seq_length=MAX_SEQ_LENGTH,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK * 2,
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model ready. Trainable params: {trainable:,}")

## 8. GRPO training functions

In [ ]:
def _apply_template(tokenizer, messages, device):
    """Apply chat template; handles Transformers 4.x (tensor) and 5.x (BatchEncoding)."""
    out = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
        return_tensors="pt",
    )
    if hasattr(out, "input_ids"):
        out = out.input_ids
    return out.to(device)


def _generate_until_valid_json(
    model, tokenizer, prompt_ids,
    *, max_total_new_tokens, chunk_new_tokens, temperature, do_sample,
):
    """Generate in chunks until the output contains a parseable JSON action."""
    generated = torch.empty(0, dtype=prompt_ids.dtype, device=prompt_ids.device)
    cursor = prompt_ids
    tokens_left = max_total_new_tokens

    while tokens_left > 0:
        n = min(chunk_new_tokens, tokens_left)
        out = model.generate(
            cursor,
            attention_mask=torch.ones_like(cursor),
            max_new_tokens=n,
            temperature=temperature,
            do_sample=do_sample,
            pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
        new_ids = out[0][cursor.shape[1]:]
        if len(new_ids) == 0:
            break
        generated = torch.cat([generated, new_ids])
        cursor = torch.cat([cursor[0], new_ids]).unsqueeze(0)
        tokens_left -= len(new_ids)
        if parse_action(tokenizer.decode(generated, skip_special_tokens=True)) is not None:
            break

    return generated


def collect_episode(
    model, tokenizer,
    *, max_episode_steps, generation_max_new_tokens, temperature, verbose=False,
):
    """Roll out one episode. Returns (transitions, success)."""
    with TimetravelEnv(base_url=ENV_URL) as env:
        obs = env_reset(env)
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        transitions = []

        model.eval()
        with torch.inference_mode():
            for step in range(max_episode_steps):
                messages.append({"role": "user", "content": obs_to_text(obs, step + 1)})
                prompt_ids = _apply_template(tokenizer, messages, model.device)
                action_ids = _generate_until_valid_json(
                    model, tokenizer, prompt_ids,
                    max_total_new_tokens=generation_max_new_tokens,
                    chunk_new_tokens=min(32, generation_max_new_tokens),
                    temperature=temperature,
                    do_sample=True,
                )
                action_text = tokenizer.decode(action_ids, skip_special_tokens=True).strip()
                action = parse_action(action_text) or {"thinking": "invalid", "action": "abandon", "args": {}}

                obs = env_step(env, format_action(action))

                if verbose:
                    print(f"  step {step+1}: pos={obs['position']} "
                          f"action={action.get('action')} reward={obs['reward']:.2f} "
                          f"budget={obs['budget_remaining']}")

                messages.append({"role": "assistant", "content": format_action(action)})
                transitions.append((prompt_ids[0].cpu(), action_ids.cpu(), float(obs["reward"])))

                # Truncate message history on branch (time rewind)
                if action.get("action") == "branch":
                    try:
                        ago = int(action.get("args", {}).get("ago", 0))
                    except (TypeError, ValueError):
                        ago = 0
                    if ago > 0:
                        keep = 1 + 2 * (step + 1 - ago)
                        messages = messages[:max(keep, 1)]

                if obs["done"]:
                    break

        model.train()
        return transitions, infer_success(obs)


def compute_return(transitions) -> float:
    return sum(r for _, _, r in transitions)


def policy_loss(model, prompt_ids, action_ids, advantage: float):
    """NLL of action tokens weighted by GRPO advantage."""
    input_ids = torch.cat([prompt_ids, action_ids]).unsqueeze(0).to(model.device)
    labels = torch.full_like(input_ids, -100)
    labels[0, len(prompt_ids):] = action_ids
    return model(input_ids=input_ids, labels=labels).loss * advantage


def evaluate_model(model, tokenizer, *, num_episodes, max_episode_steps, max_new_tokens):
    successes = 0
    branch_used = 0

    model.eval()
    with torch.inference_mode():
        for _ in range(num_episodes):
            with TimetravelEnv(base_url=ENV_URL) as env:
                obs = env_reset(env)
                messages = [{"role": "system", "content": SYSTEM_PROMPT}]
                used_branch = False

                for step in range(max_episode_steps):
                    messages.append({"role": "user", "content": obs_to_text(obs, step + 1)})
                    prompt_ids = _apply_template(tokenizer, messages, model.device)
                    action_ids = _generate_until_valid_json(
                        model, tokenizer, prompt_ids,
                        max_total_new_tokens=max_new_tokens,
                        chunk_new_tokens=min(32, max_new_tokens),
                        temperature=0.0,
                        do_sample=False,
                    )
                    action_text = tokenizer.decode(action_ids, skip_special_tokens=True).strip()
                    action = parse_action(action_text)
                    if action is None:
                        action = {"thinking": "invalid", "action": "abandon", "args": {}}

                    messages.append({"role": "assistant", "content": format_action(action)})
                    obs = env_step(env, format_action(action))

                    if action.get("action") == "branch":
                        used_branch = True
                    if obs["done"]:
                        break

                successes += int(infer_success(obs))
                branch_used += int(used_branch)

    return {
        "episodes": num_episodes,
        "success_rate": successes / num_episodes,
        "branch_rate": branch_used / num_episodes,
    }


print("GRPO functions ready.")

## 9. (Optional) W&B login

In [ ]:
try:
    import wandb
    wandb.login()  # prompts for API key in Colab
    wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME)
    use_wandb = True
    print("W&B active:", wandb.run.url)
except Exception as e:
    use_wandb = False
    print("Skipping W&B:", e)

## 10. Training loop

Custom GRPO loop (Prime Intellect style):
1. Collect `EPISODES_PER_STEP × NUM_GENERATIONS` rollouts
2. Normalize returns within each group → advantages
3. Loss = mean NLL of action tokens × advantage
4. One AdamW step per train step

In [ ]:
import time
from pathlib import Path
from torch.optim import AdamW
from torch.nn.utils import clip_grad_norm_

out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
metrics_log = []

print("Starting GRPO training...\n")
model.train()

for train_step in range(NUM_TRAIN_STEPS):
    step_start = time.perf_counter()
    optimizer.zero_grad(set_to_none=True)

    step_successes = 0
    step_returns = []
    total_loss_value = 0.0
    num_transitions = 0

    for episode_idx in range(EPISODES_PER_STEP):
        group_rollouts = []

        for gen_idx in range(NUM_GENERATIONS):
            verbose = (train_step == 0 and episode_idx == 0 and gen_idx == 0)
            transitions, success = collect_episode(
                model, tokenizer,
                max_episode_steps=MAX_EPISODE_STEPS,
                generation_max_new_tokens=GEN_MAX_NEW_TOKENS,
                temperature=TEMPERATURE,
                verbose=verbose,
            )
            ret = compute_return(transitions)
            group_rollouts.append((transitions, ret, success))
            step_successes += int(success)

        # Normalize returns → advantages
        returns = [r for _, r, _ in group_rollouts]
        mean_ret = sum(returns) / len(returns)
        std_ret  = (sum((r - mean_ret) ** 2 for r in returns) / len(returns)) ** 0.5 + 1e-8
        advantages = [(r - mean_ret) / std_ret for r in returns]

        print(f"  [grpo] step={train_step} ep={episode_idx} "
              f"returns={[round(r,3) for r in returns]} "
              f"mean={mean_ret:.3f} std={std_ret:.4f}")
        step_returns.extend(returns)

        # Accumulate gradients
        for (transitions, _, _), advantage in zip(group_rollouts, advantages):
            for prompt_ids, action_ids, _ in transitions:
                if len(action_ids) == 0:
                    continue
                if len(prompt_ids) + len(action_ids) > MAX_SEQ_LENGTH:
                    continue  # skip sequences that exceed model max length
                loss = policy_loss(model, prompt_ids, action_ids, advantage)
                loss.backward()
                total_loss_value += float(loss.detach().item())
                num_transitions += 1

    # Optimizer step
    if num_transitions > 0:
        for p in model.parameters():
            if p.grad is not None:
                p.grad.mul_(1.0 / num_transitions)
        grad_norm = clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
        optimizer.step()

    success_rate = step_successes / (EPISODES_PER_STEP * NUM_GENERATIONS)
    avg_return   = sum(step_returns) / len(step_returns)
    loss_value   = total_loss_value / max(num_transitions, 1)
    step_ms      = (time.perf_counter() - step_start) * 1000

    row = {
        "step": train_step,
        "success_rate": success_rate,
        "avg_return": avg_return,
        "loss": loss_value,
        "num_transitions": num_transitions,
    }

    if EVAL_EVERY > 0 and (train_step % EVAL_EVERY == 0):
        eval_metrics = evaluate_model(
            model, tokenizer,
            num_episodes=EVAL_EPISODES,
            max_episode_steps=MAX_EPISODE_STEPS,
            max_new_tokens=GEN_MAX_NEW_TOKENS,
        )
        row.update({f"eval_{k}": v for k, v in eval_metrics.items()})
        print(f"[eval] success_rate={eval_metrics['success_rate']:.0%} "
              f"branch_rate={eval_metrics['branch_rate']:.0%}")

    metrics_log.append(row)

    if use_wandb:
        wandb.log(row, step=train_step)

    if train_step % 10 == 0:
        print(f"step {train_step:4d} | success={success_rate:.0%} | "
              f"avg_return={avg_return:.3f} | loss={loss_value:.4f} | {step_ms:.0f}ms")

    if SAVE_EVERY > 0 and train_step > 0 and train_step % SAVE_EVERY == 0:
        ckpt = out_dir / f"checkpoint_step_{train_step}"
        model.save_pretrained(str(ckpt))
        tokenizer.save_pretrained(str(ckpt))
        print(f"Checkpoint saved: {ckpt}")

print("Training complete.")

## 11. Save final adapter

In [ ]:
final_dir = out_dir / "final_adapter"
model.save_pretrained(str(final_dir))
tokenizer.save_pretrained(str(final_dir))
print(f"Adapter saved to: {final_dir}")

if use_wandb:
    wandb.finish()

## 12. Plot training curves

In [ ]:
import matplotlib.pyplot as plt

steps   = [r["step"]         for r in metrics_log]
success = [r["success_rate"] for r in metrics_log]
returns = [r["avg_return"]   for r in metrics_log]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(steps, success); ax1.set_title("Success Rate"); ax1.set_ylim(0, 1)
ax1.set_xlabel("Train Step"); ax1.set_ylabel("Rate")
ax2.plot(steps, returns);  ax2.set_title("Avg Return")
ax2.set_xlabel("Train Step"); ax2.set_ylabel("Return")
plt.tight_layout()
plt.savefig(str(out_dir / "training_curves.png"), dpi=150)
plt.show()